# Keras/TensorFlow — Chapter 8: Evaluate the Performance of Deep Learning Models


## 1. Automatic validation set

- `model.fit(X, Y, validation_split=0.33, ...)` — Keras tự trích 33% dữ liệu **train** ra làm validation, tự tách lại **mỗi epoch giống nhau** (không xáo trộn lại giữa các epoch), báo cáo cả `loss/acc` (train) và `val_loss/val_acc` (validation) song song mỗi epoch.

## 2. Manual validation set

- Tách tường minh bằng `train_test_split()` trước, truyền vào `fit(..., validation_data=(X_test, y_test))`. Khác `validation_split` ở chỗ **kiểm soát được chính xác** tập nào dùng làm validation (VD giữ đúng tỉ lệ lớp bằng `stratify=`), thay vì để Keras tự cắt theo thứ tự dữ liệu.

## 3. K-fold cross-validation — "gold standard"

- Đây là **gold standard** để đánh giá model, ít dùng trong deep learning vì: **chi phí tính toán** — phải train lại toàn bộ model k lần, tốn gấp k lần thời gian so với 1 lần train thông thường.
- Cross-validation dùng để đánh giá **một thiết kế model** (VD 3 layer vs 4 layer), không phải một model cụ thể đã fit — nếu chỉ dùng 1 tập dữ liệu để fit rồi so sánh thì kết quả có thể chỉ phản ánh may rủi của riêng tập đó (overfitting vào cách chia).
- `StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)` — 10 fold, tỉ lệ lớp đều nhau.
- **Bắt buộc tạo model mới hoàn toàn mỗi fold** (`model = Sequential()` nằm trong vòng lặp `for train, test in kfold.split(...)`).

## 4. Kết quả thật từ sách

10 fold cho accuracy: `77.92%, 68.83%, 72.73%, 64.94%, 77.92%, 35.06%, 74.03%, 68.83%, 34.21%, 72.37%` → trung bình **64.68% ± 15.50%**.

PyTorch cho std chỉ **3.30%**, ở đây std lên tới **15.50%** — độ lệch chuẩn rất lớn, chủ yếu do 2 fold tụt hẳn xuống ~35% (gần bằng đoán ngẫu nhiên với bài toán nhị phân).


## 5. Vận dụng


**8.1** — Đánh giá bằng automatic validation set (validation_split)

In [1]:
# MLP with automatic validation set
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy
# fix random seed for reproducibility
numpy.random.seed(7)
# load pima indians dataset
dataset = numpy.loadtxt("pima-indians-diabetes.csv", delimiter=",")
# split into input (X) and output (Y) variables
X = dataset[:,0:8]
Y = dataset[:,8]
# create model
model = Sequential()
model.add(Dense(12, input_shape=(8,), activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
# Compile model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# Fit the model
model.fit(X, Y, validation_split=0.33, epochs=150, batch_size=10)


2026-09-06 06:52:49.393659: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 06:52:49.471771: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 06:52:49.471813: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 06:52:49.473560: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 06:52:49.482608: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 06:52:49.483271: I tensorflow/core/platform/cpu_feature_guard.cc:1

Epoch 1/150
52/52 [==============================] - 1s 4ms/step - loss: 4.9858 - accuracy: 0.5973 - val_loss: 3.3661 - val_accuracy: 0.5236
Epoch 2/150
52/52 [==============================] - 0s 2ms/step - loss: 3.0566 - accuracy: 0.5467 - val_loss: 2.3056 - val_accuracy: 0.5236
Epoch 3/150
52/52 [==============================] - 0s 2ms/step - loss: 2.1404 - accuracy: 0.5409 - val_loss: 1.7276 - val_accuracy: 0.5433
Epoch 4/150
52/52 [==============================] - 0s 3ms/step - loss: 1.5602 - accuracy: 0.5759 - val_loss: 1.2553 - val_accuracy: 0.5669
Epoch 5/150
52/52 [==============================] - 0s 3ms/step - loss: 1.2499 - accuracy: 0.6148 - val_loss: 1.0436 - val_accuracy: 0.6575
Epoch 6/150
52/52 [==============================] - 0s 2ms/step - loss: 1.0752 - accuracy: 0.6284 - val_loss: 0.9070 - val_accuracy: 0.5787
Epoch 7/150
52/52 [==============================] - 0s 2ms/step - loss: 0.9027 - accuracy: 0.6595 - val_loss: 0.8127 - val_accuracy: 0.6378
Epoch 8/150
5

**8.2** — Đánh giá bằng manual validation set (train_test_split)

In [2]:
# MLP with manual validation set
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
import numpy
# fix random seed for reproducibility
seed = 7
numpy.random.seed(seed)
# load pima indians dataset
dataset = numpy.loadtxt("pima-indians-diabetes.csv", delimiter=",")
# split into input (X) and output (Y) variables
X = dataset[:,0:8]
Y = dataset[:,8]
# split into 67% for train and 33% for test
X_train, X_test, y_train, y_test = train_test_split(X, Y,
                                                    test_size=0.33, random_state=seed)
# create model
model = Sequential()
model.add(Dense(12, input_shape=(8,), activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
# Compile model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# Fit the model
model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=150, batch_size=10)


Epoch 1/150
52/52 [==============================] - 1s 4ms/step - loss: 2.4641 - accuracy: 0.5214 - val_loss: 0.9909 - val_accuracy: 0.5354
Epoch 2/150
52/52 [==============================] - 0s 2ms/step - loss: 1.0956 - accuracy: 0.5331 - val_loss: 0.9439 - val_accuracy: 0.5472
Epoch 3/150
52/52 [==============================] - 0s 2ms/step - loss: 0.9912 - accuracy: 0.5467 - val_loss: 0.8678 - val_accuracy: 0.5512
Epoch 4/150
52/52 [==============================] - 0s 2ms/step - loss: 0.8914 - accuracy: 0.5681 - val_loss: 0.7885 - val_accuracy: 0.5906
Epoch 5/150
52/52 [==============================] - 0s 2ms/step - loss: 0.8230 - accuracy: 0.5875 - val_loss: 0.7209 - val_accuracy: 0.5984
Epoch 6/150
52/52 [==============================] - 0s 2ms/step - loss: 0.8147 - accuracy: 0.6051 - val_loss: 0.7295 - val_accuracy: 0.6024
Epoch 7/150
52/52 [==============================] - 0s 2ms/step - loss: 0.7407 - accuracy: 0.6109 - val_loss: 0.6935 - val_accuracy: 0.6063
Epoch 8/150
5

**8.3** — Đánh giá bằng 10-fold cross-validation (StratifiedKFold)

In [3]:
# MLP for Pima Indians Dataset with 10-fold cross validation
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import StratifiedKFold
import numpy as np
# fix random seed for reproducibility
seed = 7
np.random.seed(seed)
# load pima indians dataset
dataset = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
# split into input (X) and output (Y) variables
X = dataset[:,0:8]
Y = dataset[:,8]
# define 10-fold cross validation test harness
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
cvscores = []
for train, test in kfold.split(X, Y):
    # create model
    model = Sequential()
    model.add(Dense(12, input_shape=(8,), activation='relu'))
    model.add(Dense(8, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    # Fit the model
    model.fit(X[train], Y[train], epochs=150, batch_size=10, verbose=0)
    # evaluate the model
    scores = model.evaluate(X[test], Y[test], verbose=0)
    print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
    cvscores.append(scores[1] * 100)

print("%.2f%% (+/- %.2f%%)" % (np.mean(cvscores), np.std(cvscores)))


accuracy: 67.53%
accuracy: 72.73%
accuracy: 77.92%
accuracy: 75.32%
accuracy: 70.13%
accuracy: 67.53%
accuracy: 80.52%
accuracy: 75.32%
accuracy: 76.32%
accuracy: 78.95%
74.23% (+/- 4.37%)
